In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

In [2]:
gt_videos_path = Path(
    "../benchmark/queries_and_videos_test.json"
)
retrieved_results_path = Path(
    "../models/llm/results/Qwen3-8B_retrieval.json"
)  # Path to the JSON file create by run.py

In [3]:
with open(gt_videos_path, "r") as f:
    gt_videos_cat = json.load(f)

gt_videos = {}
for cat in gt_videos_cat:
    
    for query, video_list  in gt_videos_cat[cat].items():
        if "<vid>" in query:
            continue
        else:
            gt_videos[query] = video_list

with open(retrieved_results_path, "r") as f:
    pred_videos = json.load(f)

In [13]:
gt_videos

{'An animal doing any other activity than foraging.': ['S3_C3_E501_V0277',
  'S3_C3_E545_V0416',
  'S3_C3_E525_V0340',
  'S3_C3_E515_V0313',
  'S3_C3_E524_V0327',
  'S3_C3_E545_V0395',
  'S3_C3_E501_V0280',
  'S3_C3_E545_V0394',
  'S3_C3_E525_V0337',
  'S3_C3_E545_V0406',
  'S3_C3_E545_V0424',
  'S3_C3_E545_V0422',
  'S3_C3_E525_V0336',
  'S3_C3_E545_V0392',
  'S3_C3_E545_V0396',
  'S3_C3_E545_V0397',
  'S3_C3_E545_V0433',
  'S3_C3_E545_V0381',
  'S3_C3_E545_V0393',
  'S3_C3_E524_V0332',
  'S3_C3_E545_V0417',
  'S3_C3_E545_V0383',
  'S3_C3_E545_V0386',
  'S3_C3_E545_V0411',
  'S3_C3_E545_V0389',
  'S3_C3_E524_V0331',
  'S3_C3_E545_V0399',
  'S3_C3_E545_V0418',
  'S3_C3_E545_V0404',
  'S3_C3_E545_V0431',
  'S3_C3_E545_V0400',
  'S3_C3_E524_V0330',
  'S3_C3_E501_V0278',
  'S3_C3_E545_V0415',
  'S3_C3_E525_V0342',
  'S3_C3_E545_V0398',
  'S3_C3_E545_V0437',
  'S3_C3_E545_V0402',
  'S3_C3_E407_V0059',
  'S3_C3_E545_V0382',
  'S3_C3_E524_V0325',
  'S3_C3_E525_V0344',
  'S3_C3_E545_V0410',
 

In [22]:
results_dict = {"queries": [], "union": [], "intersection": [], "IoU": [], "F1": []}
all_gt_videos = set([v for q in gt_videos for v in gt_videos[q]])
all_pred_videos = set([v for q in pred_videos for v in pred_videos[q]])
for q in gt_videos:
    print(q)
    gt_videos_q = gt_videos[q]
    ass_videos_q = pred_videos[q]
    union = len(set(gt_videos_q).union(set(ass_videos_q)))
    intersection = len(set(gt_videos_q).intersection(set(ass_videos_q)))
    
    tp = len([v for v in ass_videos_q if v in gt_videos_q])
    
    tn = len([v for v in (set(all_gt_videos) - set(gt_videos_q)) if v not in ass_videos_q])
    
    fp = len([v for v in ass_videos_q if v not in gt_videos_q])
    
    fn = len([v for v in gt_videos_q if (v not in  ass_videos_q)])
    
    if 2*tp + fp + fn == 0:
        f1_score = np.nan
    else:
        f1_score = 2*tp/(2*tp + fp + fn)
    print("\tnb gt: \t", len(gt_videos_q))
    print("\tnb pred: \t", len(ass_videos_q))
    print("\tnb union: \t", union)
    print("\tnb intersection: \t", intersection)
    print("\tf1_score: \t", f1_score)
    if union != 0:
        IoU = intersection / union
    else:
        IoU = np.nan

    print("\tIoU: \t", IoU)

    results_dict["queries"].append(q)
    results_dict["union"].append(union)
    results_dict["intersection"].append(intersection)
    results_dict["IoU"].append(IoU)
    results_dict["F1"].append(f1_score)

    results_df = pd.DataFrame.from_dict(results_dict)

An animal doing any other activity than foraging.
	nb gt: 	 294
	nb pred: 	 721
	nb union: 	 721
	nb intersection: 	 294
	f1_score: 	 0.5793103448275863
	IoU: 	 0.4077669902912621
An animal that is neither a red deer nor a roe deer.
	nb gt: 	 22
	nb pred: 	 720
	nb union: 	 720
	nb intersection: 	 22
	f1_score: 	 0.05929919137466307
	IoU: 	 0.030555555555555555
An animal running.
	nb gt: 	 66
	nb pred: 	 66
	nb union: 	 66
	nb intersection: 	 66
	f1_score: 	 1.0
	IoU: 	 1.0
An animal bathing.
	nb gt: 	 17
	nb pred: 	 17
	nb union: 	 17
	nb intersection: 	 17
	f1_score: 	 1.0
	IoU: 	 1.0
A roe deer grazing.
	nb gt: 	 10
	nb pred: 	 10
	nb union: 	 10
	nb intersection: 	 10
	f1_score: 	 1.0
	IoU: 	 1.0
An animal browsing.
	nb gt: 	 6
	nb pred: 	 6
	nb union: 	 6
	nb intersection: 	 6
	f1_score: 	 1.0
	IoU: 	 1.0
A female adult roe deer sniffing.
	nb gt: 	 0
	nb pred: 	 0
	nb union: 	 0
	nb intersection: 	 0
	f1_score: 	 nan
	IoU: 	 nan
A juvenile red deer scratching its body.
	nb gt: 	 7

In [23]:
results_df.sort_values("IoU", ascending=False, inplace=True)

In [24]:
results_df["IoU"].describe()

count    80.000000
mean      0.880738
std       0.267867
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000
Name: IoU, dtype: float64

In [25]:
results_df["F1"].describe()

count    80.000000
mean      0.903396
std       0.239446
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000
Name: F1, dtype: float64

In [32]:
pd.set_option('display.max_colwidth', 90)
results_df[results_df["F1"] < 1]

,queries,union,intersection,IoU,F1
78,A juvenile red deer doing at least one different action than an adult female red deer.,35,34,0.971429,0.985507
76,A juvenile red deer doing the exact same activities as an adult female red deer.,33,32,0.969697,0.984615
89,An animal in vigilance while the weather is clear or sunny.,133,122,0.917293,0.956863
61,A video of two red deer.,70,59,0.842857,0.914729
30,An animal bathing while grooming.,6,5,0.833333,0.909091
106,An animal reacting to a camera and then running away.,12,9,0.750000,0.857143
110,A single adult red deer only foraging.,535,351,0.656075,0.792325
43,An adult red deer laying down while participating in courtship.,9,5,0.555556,0.714286
109,An animal reacting to the camera while the weather is clear or sunny.,45,24,0.533333,0.695652
104,An animal reacting to a camera and then foraging.,22,11,0.500000,0.666667


In [36]:
print("def check_file(json_file):\n    individual_tracks = get_tracks_from_json(json_file)\n    unique_activities = get_unique_activities_from_tracks(individual_tracks)\n    return len(unique_activities - {Activity.FORAGING}) > 0")

def check_file(json_file):
    individual_tracks = get_tracks_from_json(json_file)
    unique_activities = get_unique_activities_from_tracks(individual_tracks)
    return len(unique_activities - {Activity.FORAGING}) > 0


In [10]:
results_df.to_csv("../models/llm/results/Qwen3-8B_retrieval.csv", index=False)